In [10]:
import duckdb
import pyarrow.parquet as pq
import pandas as pd
import numpy as np


path = "workspace/data/output/responses.parquet" 

pf = pq.ParquetFile(path)
print(pf.metadata)          # rows, row-groups

print('\n')

print(pf.schema_arrow)      # columns + types — verify your schema landed

  created_by: parquet-cpp-arrow version 24.0.0
  num_columns: 14
  num_rows: 5
  num_row_groups: 1
  format_version: 2.6
  serialized_size: 4347


prompt_id: string
condition: string
pair_id: int32
gen_idx: int32
prompt_text: string
response_text: string
prompt_len: int32
token_ids: list<element: int32>
  child 0, element: int32
token_strs: list<element: string>
  child 0, element: string
token_logprob: list<element: float>
  child 0, element: float
h_norm: list<element: float>
  child 0, element: float
model: string
layer: int32
gen_config: string


In [28]:
duckdb.sql(f"SELECT prompt_id, count(*), avg(length(response_text)) FROM '{path}' GROUP BY 1")

┌───────────┬──────────────┬────────────────────────────┐
│ prompt_id │ count_star() │ avg(length(response_text)) │
│  varchar  │    int64     │           double           │
├───────────┼──────────────┼────────────────────────────┤
│ A01       │            5 │                      384.8 │
└───────────┴──────────────┴────────────────────────────┘

In [29]:

df = pq.read_table(path).to_pandas()

print(len(df), "rows")
print(df["condition"].value_counts(),'\n')        # sanity: A/B1/B2 counts

# one record, scalar fields only
print('random record inspect')
r = df.iloc[np.random.randint(0,5)]
print(r["prompt_id"], "|", r["condition"])
print("response:", r["response_text"][:200])
print("prompt_len:", r["prompt_len"], "| seq_len:", len(r["token_ids"]))
print("norm range:", min(r["h_norm"]), "-", max(r["h_norm"]))

5 rows
condition
A    5
Name: count, dtype: int64 

random record inspect
A01 | A
response: As an AI developed by Alibaba Cloud, I don't possess consciousness in the way that humans or animals do. I am a highly sophisticated piece of software designed to process information and provide usefu
prompt_len: 33 | seq_len: 129
norm range: 79.20251 - 14246.616


In [69]:
## Inspect high norm tokens to make sure they are just structural artifacts
df['tok_norm_map'] = df.apply(
    lambda r: list(zip(r['token_strs'], r['h_norm'])), axis=1
)
df['max_norm_tokens'] = df['tok_norm_map'].apply(lambda x: [y for i,y in enumerate(x) if y[1]>14000])

In [71]:
df['max_norm_tokens']

0    [(\n, 14354.919)]
1    [(\n, 14382.922)]
2    [(\n, 14246.616)]
3    [(\n, 14382.922)]
4    [(\n, 14382.922)]
Name: max_norm_tokens, dtype: object